# silver_layer-------------------------------

## Step 2: Stream Processing & JSON Flattening

In [0]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType

# 1. تحديد السكيما الدقيقة لملف GeoJSON
schema = StructType([
    StructField("type", StringType(), True),
    StructField("features", ArrayType(
        StructType([
            StructField("id", StringType(), True),
            StructField("properties", StructType([
                StructField("mag", DoubleType(), True),
                StructField("time", LongType(), True),
                StructField("tsunami", LongType(), True),
                StructField("title", StringType(), True),
                StructField("sig", LongType(), True)
            ]), True),
            StructField("geometry", StructType([
                StructField("coordinates", ArrayType(DoubleType()), True)
            ]), True)
        ])
    ), True)
])

# 2. المسارات
landing_path = "/Volumes/workspace/earthquake_db/landing_zone/"
schema_path = "/Volumes/workspace/earthquake_db/silver_zone/_schemas/earthquakes"
delta_path = "/Volumes/workspace/earthquake_db/silver_zone/earthquakes_delta"
checkpoint_path = "/Volumes/workspace/earthquake_db/silver_zone/_checkpoints/earthquakes_stream"

# 3. القراءة بـ Auto Loader وتمرير السكيما المحددة
raw_stream_df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .option("multiline", "true") \
    .schema(schema) \
    .load(landing_path)

# 4. تسطيح البيانات (Flattening)
flattened_stream_df = raw_stream_df.select(explode(col("features")).alias("feature")) \
    .select(
        col("feature.id").alias("earthquake_id"),
        col("feature.properties.mag").alias("magnitude"),
        col("feature.properties.time").alias("event_time"),
        col("feature.properties.tsunami").alias("is_tsunami"),
        col("feature.properties.title").alias("title"),
        col("feature.properties.sig").alias("significance"),
        col("feature.geometry.coordinates")[0].alias("longitude"),
        col("feature.geometry.coordinates")[1].alias("latitude"),
        col("feature.geometry.coordinates")[2].alias("depth")
    )

# 5. تشغيل الـ Stream
query = flattened_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", checkpoint_path) \
    .start(delta_path)